# Problem Set 3. Text I/II: classical representations, classification & embeddings

**Course:** Analysis of Unstructured Data  
**Instructor:** Janusz Szwabiński  
**Goal:** Move from sparse count vectors to dense distributional semantics.  

---


In [ ]:
# Setup environment & imports
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'


### Problem 1 [Theory] (TF-IDF)

**Task:**
Derive the TF-IDF weight $w_{t,d} = \mathrm{tf}_{t,d} \cdot \log \frac{N}{\mathrm{df}_t}$. Give an information-theoretic reading of the $\log(N/\mathrm{df}_t)$ term and explain why raw term frequency alone is a poor weighting.

---

### Assumptions & Notation

1. **Document Collection (Corpus):** Let $\mathcal{D} = \{d_1, d_2, \dots, d_N\}$ be a corpus of $N = |\mathcal{D}|$ documents.
2. **Vocabulary:** Let $V = \{t_1, t_2, \dots, t_{|V|}\}$ be the set of unique terms occurring in $\mathcal{D}$.
3. **Raw Term Frequency:** For term $t \in V$ and document $d \in \mathcal{D}$, let $f_{t,d} \in \mathbb{N}_0$ denote the number of times $t$ occurs in $d$.
4. **Normalized Term Frequency ($\mathrm{tf}_{t,d}$):** Within-document frequency normalized to capture relative prominence, standardly defined as:
   $$\mathrm{tf}_{t,d} = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}} \quad \text{or sublinearly as} \quad \mathrm{tf}_{t,d} = \begin{cases} 1 + \log f_{t,d}, & \text{if } f_{t,d} > 0, \\ 0, & \text{otherwise.} \end{cases}$$
5. **Document Frequency ($\mathrm{df}_t$):** The number of documents in $\mathcal{D}$ that contain term $t$:
   $$\mathrm{df}_t = |\{d \in \mathcal{D} : f_{t,d} > 0\}|, \quad \text{with } 1 \le \mathrm{df}_t \le N.$$
6. **Probabilistic Framework:** Consider the experiment of drawing a document $D$ uniformly at random from $\mathcal{D}$, such that $\mathbb{P}(D = d) = \frac{1}{N}$ for all $d \in \mathcal{D}$.

---

### Formal Proposition

1. **Information-Theoretic Surprisal of a Term:**
   Under the uniform document distribution, the probability that a randomly sampled document contains term $t$ is:
   $$\mathbb{P}(t \in D) = \frac{\mathrm{df}_t}{N}$$
   The Shannon self-information (surprisal) associated with observing term $t$ in a document is:
   $$I(t) = -\log \mathbb{P}(t \in D) = -\log \left(\frac{\mathrm{df}_t}{N}\right) = \log \left(\frac{N}{\mathrm{df}_t}\right) = \mathrm{idf}_t$$

2. **Compound TF-IDF Weight:**
   The composite weight $w_{t,d}$ measures the joint importance of term $t$ to document $d$ as the product of its local within-document salience and its global collection-wide discriminative information:
   $$w_{t,d} = \mathrm{tf}_{t,d} \cdot \mathrm{idf}_t = \mathrm{tf}_{t,d} \cdot \log \left(\frac{N}{\mathrm{df}_t}\right)$$

---

### Step-by-Step Derivation & Information-Theoretic Reading

#### Step 1: Probability of Document-Term Occurrence
Let $X_t$ be a binary Bernoulli random variable indicating whether term $t$ is present in a randomly drawn document $D \sim \mathrm{Uniform}(\mathcal{D})$:
$$X_t = \mathbb{I}(t \in D) = \begin{cases} 1, & \text{if } t \in D, \\ 0, & \text{if } t \notin D. \end{cases}$$
The empirical probability of occurrence across the collection is:
$$\mathbb{P}(X_t = 1) = \sum_{d \in \mathcal{D}} \mathbb{P}(t \in D \mid D = d) \mathbb{P}(D = d) = \sum_{d \in \mathcal{D} : t \in d} \frac{1}{N} = \frac{\mathrm{df}_t}{N}$$

#### Step 2: Information-Theoretic Derivation of IDF (Shannon Surprisal)
In Shannon information theory, the information content (or self-information / surprisal) of an event $E$ having probability $\mathbb{P}(E)$ is defined as:
$$h(E) = \log \frac{1}{\mathbb{P}(E)} = -\log \mathbb{P}(E)$$
- **High surprisal (rare terms):** If a term $t$ appears in very few documents ($\mathrm{df}_t \ll N$), $\mathbb{P}(t \in D)$ is very small. Observing $t$ in a document is an unexpected, highly informative event. It significantly reduces our uncertainty about the document's specific subject matter. The surprisal is large:
  $$I(t) = \log \frac{N}{\mathrm{df}_t} \gg 0$$
- **Low surprisal (common terms):** If a term appears in almost every document ($\mathrm{df}_t \approx N$), $\mathbb{P}(t \in D) \approx 1$. Knowing that $t$ is present imparts negligible new information (near zero surprisal), as virtually all documents share this property:
  $$I(t) = \log \frac{N}{\mathrm{df}_t} \approx \log 1 = 0$$

Thus, the inverse document frequency $\mathrm{idf}_t = \log(N/\mathrm{df}_t)$ is mathematically identical to the Shannon self-information of the occurrence event $t \in D$.

#### Step 3: Divergence from Background Language Model
An alternative information-theoretic derivation arises from language modeling (Croft & Harper 1979; Roelleke & Wang 2008). 
Let $M_0$ denote the collection background model where terms are distributed according to null occurrence probability $p_0(t) = \frac{\mathrm{df}_t}{N}$, and let $M_d$ denote a document model. 
The Kullback-Leibler (KL) divergence contribution measuring how much term $t$ distinguishes document $d$ from the uninformative background collection is:
$$D_{\mathrm{KL}}(M_d \parallel M_0) = \sum_{t \in V} p_d(t) \log \frac{p_d(t)}{p_0(t)}$$
When term $t$ is present in $d$, the factor $\log \frac{1}{p_0(t)} = \log \frac{N}{\mathrm{df}_t}$ acts as the fundamental multiplier weighting the term's divergence from the global background distribution.

#### Step 4: Critique: Why Raw Term Frequency Alone is a Poor Weighting
Relying solely on the raw count $f_{t,d}$ fails for three fundamental reasons:

1. **Document Length Bias:**
   In raw frequency representations, long documents receive systematically larger values merely because they contain more total tokens, regardless of whether they have greater semantic specificity. Normalizing to $\mathrm{tf}_{t,d}$ corrects for document length.
2. **Zipf's Law and the Stopword Confound:**
   According to Zipf's Law, the frequency of any word is inversely proportional to its frequency rank ($f(r) \propto 1/r$). A handful of functional and grammatical words (*"the"*, *"and"*, *"of"*, *"is"*, *"in"*) account for a massive fraction of all tokens in any text corpus. 
   If raw counts $f_{t,d}$ are used to form bag-of-words vectors:
   $$\mathbf{d}_i^\top \mathbf{d}_j = \sum_{t \in V} f_{t,d_i} f_{t,d_j}$$
   this inner product (and the resulting cosine similarity) is overwhelmingly dominated by shared occurrences of ubiquitous stopwords, entirely masking the topical keywords that distinguish one subject from another. Multiplying by $\log(N/\mathrm{df}_t)$ naturally attenuates or eliminates these uninformative terms because $\log(N/N) = 0$.
3. **Diminishing Marginal Informativeness (Sublinear Utility):**
   A term that occurs 20 times in a document is not 20 times more indicative of that topic than a term appearing once or twice. The first few occurrences confirm topicality; subsequent occurrences provide diminishing marginal evidence. Raw linear counts severely overweight repetitive terms.

Combining within-document prominence $\mathrm{tf}_{t,d}$ with collection specificity $\log(N/\mathrm{df}_t)$ yields an optimal balance between local relevance and global discrimination.

---

### Sanity Checks & Edge Cases

1. **Omnipresent Stopword ($\mathrm{df}_t = N$):**
   If a term appears in all $N$ documents:
   $$\mathrm{idf}_t = \log \left(\frac{N}{N}\right) = \log(1) = 0 \implies w_{t,d} = 0 \quad \forall d \in \mathcal{D}$$
   The term receives zero weight automatically, without needing an explicit manual stopword blacklist.
2. **Hapax Legomenon across Corpus ($\mathrm{df}_t = 1$):**
   If a unique term appears in only a single document:
   $$\mathrm{idf}_t = \log \left(\frac{N}{1}\right) = \log N$$
   This attains the maximum possible IDF value across the vocabulary, maximizing the term's ability to identify its unique source document.
3. **Smooth IDF in Practice:**
   To prevent division by zero for out-of-vocabulary terms in test sets and prevent total annihilation of terms where $\mathrm{df}_t = N$, practical packages (such as scikit-learn) introduce smoothing constants:
   $$\mathrm{idf}_t^{\text{smooth}} = \log \left(\frac{1 + N}{1 + \mathrm{df}_t}\right) + 1$$
   which guarantees $w_{t,d} > 0$ for all non-empty terms.


### Problem 2 [Theory] (Multinomial Naïve Bayes)

---
**Derivation & Comparison:**


### Problem 3 [Theory] (Skip-gram with negative sampling)

---
**Objective, Gradient & PMI Factorisation:**


### Problem 4 [Python]

**Task:**
Build a TF-IDF pipeline (spaCy/NLTK preprocessing -> TfidfVectorizer) and train both multinomial Naïve Bayes and logistic regression on a text-classification dataset (e.g. 20 Newsgroups subset). Report precision/recall/F1 and list the most informative features per class.


In [ ]:
# Problem 4 [Python]
import spacy
import nltk
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# TODO: Implement TF-IDF pipeline, train MNB & LogReg, report metrics and top features


### Problem 5 [Python]

**Task:**
Train word2vec with gensim on the same corpus and separately load pretrained GloVe. For both, return the nearest neighbours of five query words and evaluate on an analogy set (e.g. king - man + woman).


In [ ]:
# Problem 5 [Python]
import gensim
from gensim.models import Word2Vec
import gensim.downloader as api

# TODO: Train Word2Vec and evaluate queries + analogies


### Problem 6 [Analysis] (Embedding bias)

---
**Findings & Discussion:**
